# Project spoke deployment

Deploy a **Project Team (Spoke)** that connects to the Core Gateway.

## What gets deployed

| Resource | Purpose |
|----------|--------|
| AI Foundry Account | Project team's AI account |
| Foundry Project | Workspace for the team |
| Core Gateway Connection | Gateway connection to core models (API Key auth) |

## Key requirements

- **Connection Auth**: Uses API Key authentication (AAD not yet supported for APIM connections)
- **Connection Target**: Must include the API path (e.g., `https://apim.azure-api.net/openai`)
- **Static Model Discovery**: Models defined in connection metadata (no dynamic discovery endpoint needed)

> ⚠️ **Prerequisite**: Complete deployment of the Core Gateway

## Step 1: Load core configuration

In [1]:
import os, subprocess
from pathlib import Path

# Load .env from repo root
repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
env_file = repo_root / '.env'
with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

CORE_ENDPOINT = os.environ['CORE_ENDPOINT']
GATEWAY_URL = os.environ['GATEWAY_URL']
GATEWAY_KEY = os.environ['ALPHA_GATEWAY_KEY']
CHAT_MODEL = os.environ['CHAT_MODEL']

print(f"Core Endpoint: {CORE_ENDPOINT}")
print(f"Gateway URL:  {GATEWAY_URL}")
print(f"Gateway Key:  {GATEWAY_KEY[:2]}... (hidden)")
print(f"Chat Model:   {CHAT_MODEL}")

Core Endpoint: https://aif-core-c2676f.cognitiveservices.azure.com/
Gateway URL:  https://apim-foundry-c2676f.azure-api.net/openai
Gateway Key:  83... (hidden)
Chat Model:   gpt-4.1-mini


## Step 2: Set spoke variables

In [2]:
import hashlib, subprocess

# Set the team name for this spoke deployment
# Used to namespace resources and .env keys - allows multiple spokes in the same subscription
TEAM_NAME = "alpha"
TEAM_PREFIX = TEAM_NAME.upper().replace('-', '_')  # e.g. "alpha" -> "ALPHA"

# Same suffix as the core gateway deployment - derived from subscription ID
SUB_ID = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
SUFFIX = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]

SPOKE_RG = f"rg-foundry-spoke-{TEAM_NAME}-{SUFFIX}"
CORE_RG = f"rg-foundry-core-{SUFFIX}"
LOCATION = "eastus2"

print(f"Team:      {TEAM_NAME}  (env prefix: {TEAM_PREFIX}_*)")
print(f"Suffix:    {SUFFIX}")
print(f"Spoke RG:  {SPOKE_RG}")
print(f"Core RG:    {CORE_RG}")

Team:      alpha  (env prefix: ALPHA_*)
Suffix:    c2676f
Spoke RG:  rg-foundry-spoke-alpha-c2676f
Core RG:    rg-foundry-core-c2676f


## Step 3: Create resource group

In [3]:
!az group create -n "{SPOKE_RG}" -l "{LOCATION}" -o table

Location    Name
----------  -----------------------------
eastus2     rg-foundry-spoke-alpha-c2676f


## Step 4: Deploy spoke infrastructure

⏱️ Takes ~2-3 minutes

In [4]:
import subprocess, json, base64

# Get principal ID from cached JWT token (avoids graph.microsoft.com network call)
token = subprocess.run('az account get-access-token --query accessToken -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
payload = token.split('.')[1] + '=='
PRINCIPAL_ID = json.loads(base64.b64decode(payload))['oid']
print(f"Principal ID: {PRINCIPAL_ID}")
print(f"Team:         {TEAM_NAME}")
print(f"Suffix:       {SUFFIX}")

!az deployment group create -g "{SPOKE_RG}" --template-file main.bicep \
    -p deployerPrincipalId="{PRINCIPAL_ID}" \
    -p suffix="{SUFFIX}" \
    -p teamName="{TEAM_NAME}" \
    -p apimUrl="{GATEWAY_URL}" \
    -p chatModelName="{CHAT_MODEL}" \
    -p apimSubscriptionKey="{GATEWAY_KEY}" \
    -o table

Principal ID: 00000000-0000-0000-0000-000000000000
Team:         alpha
Suffix:       c2676f
=A new Bicep release is available: v0.43.8. Upgrade now by running "az bicep upgrade".
Name    State      Timestamp                         Mode         ResourceGroup
------  ---------  --------------------------------  -----------  -----------------------------
main    Succeeded  2026-05-10T08:30:29.278128+00:00  Incremental  rg-foundry-spoke-alpha-c2676f


## Step 5: Get spoke outputs

In [5]:
import subprocess, json, os
from pathlib import Path

r = subprocess.run(f'az deployment group show -g "{SPOKE_RG}" -n main --query properties.outputs -o json', shell=True, capture_output=True, text=True)
out = json.loads(r.stdout)

SPOKE_ACCOUNT = out['accountName']['value']
SPOKE_ENDPOINT = out['accountEndpoint']['value']
SPOKE_PROJECT = out['projectName']['value']
PROJECT_ENDPOINT = out['projectEndpoint']['value']
CORE_CONNECTION = out['apimConnectionName']['value']

print(f"Spoke Account:    {SPOKE_ACCOUNT}")
print(f"Spoke Endpoint:   {SPOKE_ENDPOINT}")
print(f"Project Name:     {SPOKE_PROJECT}")
print(f"Project Endpoint: {PROJECT_ENDPOINT}")
print(f"Core Connection:   {CORE_CONNECTION}")

# Merge outputs into .env using team-namespaced keys (supports multiple spokes)
repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
env_file = repo_root / '.env'

existing = {}
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            existing[k.strip()] = v.strip()

# Remove stale keys (renamed in prior versions of this notebook)
for stale in ('SPOKE_ACCOUNT', 'SPOKE_ENDPOINT', 'SPOKE_PROJECT', 'PROJECT_ENDPOINT', 'APIM_CONNECTION',
              f'{TEAM_PREFIX}_FOUNDRY_GATEWAY_CONNECTION'):
    existing.pop(stale, None)

new_values = {
    f'{TEAM_PREFIX}_FOUNDRY_ACCOUNT':          SPOKE_ACCOUNT,
    f'{TEAM_PREFIX}_FOUNDRY_ENDPOINT':         SPOKE_ENDPOINT,
    f'{TEAM_PREFIX}_FOUNDRY_PROJECT':          SPOKE_PROJECT,
    f'{TEAM_PREFIX}_FOUNDRY_PROJECT_ENDPOINT': PROJECT_ENDPOINT,
    f'{TEAM_PREFIX}_FOUNDRY_CORE_CONNECTION':   CORE_CONNECTION,
}

existing.update(new_values)
env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')

# Also update os.environ so subsequent cells in this session use the fresh values
os.environ.update(new_values)

print(f"\n✅ Outputs saved to {env_file} under {TEAM_PREFIX}_FOUNDRY_* keys")

Spoke Account:    aif-spoke-alpha-c2676f
Spoke Endpoint:   https://aif-spoke-alpha-c2676f.cognitiveservices.azure.com/
Project Name:     project-alpha-c2676f
Project Endpoint: https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Core Connection:   core-alpha

✅ Outputs saved to <repo-root>/.env under ALPHA_FOUNDRY_* keys


## Step 6: Connect to project and test agent

Now test the spoke by creating an agent that uses the APIM gateway connection.

In [6]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()

project_client = AIProjectClient(
    credential=credential,
    endpoint=PROJECT_ENDPOINT
)

openai_client = project_client.get_openai_client()

print(f"✅ Connected to Spoke Project: {SPOKE_PROJECT}")
print(f"✅ Project Endpoint: {PROJECT_ENDPOINT}")
print(f"✅ OpenAI client ready for responses API")

✅ Connected to Spoke Project: project-alpha-c2676f
✅ Project Endpoint: https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
✅ OpenAI client ready for responses API


## Step 7: Create and test Hello World agent

Create a simple agent that uses the core gateway connection.

In [7]:
from azure.ai.projects.models import PromptAgentDefinition

# Core model format: <connection-name>/<model-id>
CORE_MODEL = f"{CORE_CONNECTION}/{CHAT_MODEL}"
HELLO_AGENT_NAME = "hello-world-agent"

print(f"Creating Hello World agent...")
print(f"Using core model: {CORE_MODEL}")

hello_agent = project_client.agents.create_version(
    agent_name=HELLO_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CORE_MODEL,
        instructions="You are a friendly assistant. When someone greets you, respond with a cheerful greeting that includes the phrase 'Hello from the Spoke via APIM gateway!' somewhere in your response. Keep responses brief."
    ),
)

print(f"✅ Agent created: {hello_agent.name} v{hello_agent.version}")
print(f"   Model: {CORE_MODEL}")

Creating Hello World agent...
Using core model: core-alpha/gpt-4.1-mini
✅ Agent created: hello-world-agent v1
   Model: core-alpha/gpt-4.1-mini


Show the agent config as known to foundry project, including the model reference to the apim model

In [8]:
agent = project_client.agents.get_version(
      agent_name="hello-world-agent",
      agent_version="1"
  )
print(f"Model:        {agent.definition.model}")
print(f"Instructions: {agent.definition.instructions}")
print(f"Raw:          {agent.definition}")

Model:        core-alpha/gpt-4.1-mini
Instructions: You are a friendly assistant. When someone greets you, respond with a cheerful greeting that includes the phrase 'Hello from the Spoke via APIM gateway!' somewhere in your response. Keep responses brief.
Raw:          {'kind': 'prompt', 'model': 'core-alpha/gpt-4.1-mini', 'instructions': "You are a friendly assistant. When someone greets you, respond with a cheerful greeting that includes the phrase 'Hello from the Spoke via APIM gateway!' somewhere in your response. Keep responses brief."}


Show the foundry project connections (as currently not surfaced in any foundry next gen project UI)

In [9]:
connections = list(project_client.connections.list())
for c in connections:
    print(c)

{'name': 'core-alpha', 'id': '/subscriptions/00000000-0000-0000-0000-000000000000/resourceGroups/rg-foundry-spoke-alpha-c2676f/providers/Microsoft.CognitiveServices/accounts/aif-spoke-alpha-c2676f/projects/project-alpha-c2676f/connections/core-alpha', 'type': 'ApiManagement', 'target': 'https://apim-foundry-c2676f.azure-api.net/openai', 'isDefault': True, 'credentials': {'type': 'ApiKey'}, 'metadata': {'deploymentInPath': 'true', 'inferenceAPIVersion': '2024-10-21', 'models': '[{"name":"gpt-4.1-mini","properties":{"model":{"name":"gpt-4.1-mini","version":"","format":"OpenAI"}}}]'}}


In [10]:
# Invoke the agent using the Responses API
query = "Hello! I'm testing the BYO Gateway feature from the spoke project. Can you confirm this is working?"
print(f"User: {query}")

# Use the Responses API with agent_reference (must include version)
response = openai_client.responses.create(
    input=query,
    extra_body={
        "agent_reference": {
            "name": hello_agent.name,
            "version": hello_agent.version,
            "type": "agent_reference"
        }
    }
)

# Extract the response text
agent_response = response.output_text if hasattr(response, 'output_text') else str(response.output)

print("\n" + "="*50)
print("CONVERSATION:")
print("="*50)
print(f"\n🧑 User:\n  {query}")
print(f"\n🤖 Agent:\n  {agent_response}")

User: Hello! I'm testing the BYO Gateway feature from the spoke project. Can you confirm this is working?

CONVERSATION:

🧑 User:
  Hello! I'm testing the BYO Gateway feature from the spoke project. Can you confirm this is working?

🤖 Agent:
  Hello! Hello from the Spoke via APIM gateway! Your BYO Gateway feature seems to be working perfectly. How can I assist you further?


In [11]:
#!az group delete -n "{SPOKE_RG}" --yes --no-wait

## Done!

Your project spoke is now connected to the Core Gateway. The team can:
- Access shared models via core gateway connection
- Build agents using `<connection-name>/<model-id>` format
- Benefit from centralized rate limiting and policies
- Have their usage tracked for cost allocation

### Key concepts

| Concept | Description |
|---------|-------------|
| Gateway Model Format | `<connection-name>/<model-id>` (e.g., `core-alpha/gpt-4.1-mini`) |
| Connection Auth | API Key (stored securely in connection credentials) |
| Static Model Discovery | Models defined in connection metadata via `models` JSON string |
| PromptAgentDefinition | Defines agent with model, instructions, and tools |
| Responses API | Invoke agents via `openai_client.responses.create()` with `agent_reference` |

## Cleanup (optional)

In [12]:
# Clean up the Hello World agent
# project_client.agents.delete(agent_name=HELLO_AGENT_NAME)
# print(f"✅ Hello World agent '{HELLO_AGENT_NAME}' deleted")
# !az group delete -n "{SPOKE_RG}" --yes --no-wait